X_test = [
  [1.5, 2.5]
]
X_train = [
  [1, 2],
  [2, 3],
  [3, 4],
  [4, 5]
]
y_train = [
  0,
  0,
  1,
  1
]

In [ ]:
import numpy as np


class GaussianProcessClassifier:
    def __init__(self, length_scale: float = 1.0, sigma_f: float = 1.0,
                 noise: float = 1e-6):
        self.length_scale = length_scale
        self.sigma_f = sigma_f
        self.noise = noise
        self.X_train = None
        self.y_train = None
        self.L = None
        self.alpha = None

    def _rbf_kernel(self, X1, X2):
        # Pairwise squared distances
        X1_sq = np.sum(X1 ** 2, axis=1).reshape(-1, 1)
        X2_sq = np.sum(X2 ** 2, axis=1).reshape(1, -1)
        sq_dist = X1_sq + X2_sq - 2 * X1 @ X2.T
        scale = -0.5 / (self.length_scale ** 2)
        return (self.sigma_f ** 2) * np.exp(scale * sq_dist)

    def fit(self, X, y) -> None:
        self.X_train = np.asarray(X, dtype=float)
        self.y_train = np.asarray(y, dtype=float).reshape(-1, 1)
        K = self._rbf_kernel(self.X_train, self.X_train)
        K += self.noise * np.eye(self.X_train.shape[0])
        self.L = np.linalg.cholesky(K)
        v = np.linalg.solve(self.L, self.y_train)
        self.alpha = np.linalg.solve(self.L.T, v)

    def predict_proba(self, X):
        X_test = np.asarray(X, dtype=float)
        k_star = self._rbf_kernel(self.X_train, X_test)
        # Predictive mean
        mean = k_star.T @ self.alpha
        # Predictive variance
        v = np.linalg.solve(self.L, k_star)
        k_xx = (self.sigma_f ** 2) * np.ones(X_test.shape[0])
        var = k_xx - np.sum(v ** 2, axis=0)
        # Calibrated sigmoid
        denom = np.sqrt(1.0 + (np.pi / 8.0) * var)
        prob = 1.0 / (1.0 + np.exp(-mean.ravel() / denom))
        return prob.tolist()

# Instantiate model
model = GaussianProcessClassifier()
# Fit model
model.fit(X_train, y_train)
# Get predictions
y_test = model.predict_proba(X_test)
# Output predictions
y_test